#                                            Python project - Roei Raz - 322817081 #

## github link:  https://github.com/roeiraz12/Data-minning-project---3rd-year.git

# PART 1 #

#

## Collecting data columns from IMDb libraries ##

In [1]:
#ייבוא ספריות רלוונטיות
import urllib.request
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
from tqdm import tqdm 

## Filtering the movies as defined 

1. type= "movie"
2. start year < 2025
3. run time between 60-300
4. 'primaryTitle' starts with E/F
5. 'primaryTitle' has no special chars (as ! , and foreign languges) - my additon

In [2]:
#קריאה ופלטור ראשוני של הדאטה
df_basics = pd.read_csv(
    "title.basics.tsv.gz", 
    sep='\t', 
    na_values='\\N', 
    usecols=['tconst', 'titleType', 'primaryTitle', 'startYear', 'genres', 'runtimeMinutes'],
    low_memory=False
)

chars = r'^[a-zA-Z\s]+$'

# המרת ערכים למספרים
df_basics['startYear'] = pd.to_numeric(df_basics['startYear'], errors='coerce') 
df_basics['runtimeMinutes'] = pd.to_numeric(df_basics['runtimeMinutes'], errors='coerce')


mask = ( #התנאים שכתבנו למעלה
    (df_basics['titleType'] == 'movie') & 
    (df_basics['startYear'] < 2025) & 
    (df_basics['runtimeMinutes'] >= 60) & 
    (df_basics['runtimeMinutes'] <= 300) & 
    (df_basics['primaryTitle'].str.match(chars, na=False)) & 
    (df_basics['primaryTitle'].str.startswith(('E', 'F'), na=False))
)

df_filtered = df_basics[mask].copy()

#בחירת תאים מלאים בנתונים
df_full_values = df_filtered.dropna()

#בחירת 5000 השורות הראשונות
df_basics = df_full_values.head(5000)

df_basics.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
2174,tt0002199,movie,From the Manger to the Cross,1912.0,71.0,"Biography,Drama"
2793,tt0002820,movie,East Lynne,1913.0,69.0,Drama
2858,tt0002885,movie,From Dusk to Dawn,1913.0,90.0,Drama
6340,tt0006419,movie,El beso de la muerte,1917.0,72.0,Drama
6602,tt0006685,movie,Fires of Conscience,1916.0,60.0,Drama


## Collecting only the relevant data from the other tables by comparring the key field ('tconst')

In [3]:
#יצירת עותק
df_basics = df_filtered[['tconst', 'primaryTitle', 'startYear', 'genres', 'runtimeMinutes']].copy()

# המרת הנתונים לטייפ כמבוקש 
df_basics['genres'] = df_basics['genres'].str.split(',')
df_basics['startYear'] = pd.to_numeric(df_basics['startYear'], errors='coerce')
df_basics['runtimeMinutes'] = pd.to_numeric(df_basics['runtimeMinutes'], errors='coerce')
df_basics = df_basics.astype({'startYear': 'Int64','runtimeMinutes': 'Int64'})
df_basics.head()

,tconst,primaryTitle,startYear,genres,runtimeMinutes
2174,tt0002199,From the Manger to the Cross,1912,"[Biography, Drama]",71
2289,tt0002315,El lobo de la sierra,1912,NaN,76
2793,tt0002820,East Lynne,1913,[Drama],69
2858,tt0002885,From Dusk to Dawn,1913,[Drama],90
3910,tt0003951,Fatum,1915,NaN,70


# #

In [4]:
#טעינת הקובץ השני ושמירה רק של הסרטים הרלוונטים לנו ע"פ הסינון שכבר בוצע
df_ratings_raw = pd.read_csv("title.ratings.tsv.gz", sep='\t', usecols=['tconst', 'averageRating', 'numVotes'])
df_ratings_raw['numVotes'] = pd.to_numeric(df_ratings_raw['numVotes'], errors='coerce')
df_ratings_raw['numVotes'] = df_ratings_raw['numVotes'].astype('Int64')
df_ratings = pd.merge(df_basics[['tconst']], df_ratings_raw, on='tconst', how='inner')
df_ratings.head()

,tconst,averageRating,numVotes
0,tt0002199,5.9,712
1,tt0002820,6.2,22
2,tt0002885,5.9,119
3,tt0006419,4.1,24
4,tt0007983,6.1,283


In [5]:
final_movie_ids = set(df_basics['tconst']) #סרטים אלו עברו את הסינון לכן רק להם נמצא מידע
#קריאת הקובץ הראשונית
#מכיוון שהקובץ ארוך מאוד נעבד אותו בחלקים
chunks_prin = pd.read_csv( "title.principals.tsv.gz",  sep='\t',  na_values='\\N', chunksize=1000000, usecols=['tconst', 'nconst', 'category', 'ordering'], low_memory=False)

#נמצא את שמות חמשת השחקנים המובילים בכל סרט
processed_chunks = []
for chunk in chunks_prin:
    mask = (chunk['category'].isin(['actor', 'actress'])) & \
           (chunk['ordering'] <= 5) & \
           (chunk['tconst'].isin(final_movie_ids))
    processed_chunks.append(chunk[mask])

df_actors_long = pd.concat(processed_chunks)

# נזין את כל חמשת השמות לרשימה אחת בתוך התא
principals_df = df_actors_long.groupby('tconst')['nconst'].apply(list).reset_index()

# שמות לעמודות
principals_df.columns = ['tconst', 'lead_actors_ids']

principals_df.head()

,tconst,lead_actors_ids
0,tt0002199,"[nm0087381, nm0245769, nm0310155, nm0391220, n..."
1,tt0002820,"[nm0287085, nm0666835, nm0604658, nm0211667, n..."
2,tt0002885,"[nm0191890, nm0201679, nm0204584, nm0254556, n..."
3,tt0003951,"[nm0100479, nm0458607, nm0458607, nm0863812, n..."
4,tt0004594,"[nm0892836, nm0898147, nm0679170, nm0035412, n..."


## Now we got all the data from IMDB database

 ## Next - merging all three tables to one dataset ##

In [6]:
#איחוד שלושת הטבלאות
first = df_basics.merge(df_ratings, on='tconst', how='left')
imdb_df = first.merge(principals_df, on='tconst', how='left')
df_full_values = imdb_df.dropna() #ניקוי שורות לא מלאות
imdb_df = df_full_values.head(5000)
imdb_df.to_csv('imdb_pd.csv', index=False, encoding='utf-8-sig')
imdb_df.head()

,tconst,primaryTitle,startYear,genres,runtimeMinutes,averageRating,numVotes,lead_actors_ids
0,tt0002199,From the Manger to the Cross,1912,"[Biography, Drama]",71,5.9,712,"[nm0087381, nm0245769, nm0310155, nm0391220, n..."
2,tt0002820,East Lynne,1913,[Drama],69,6.2,22,"[nm0287085, nm0666835, nm0604658, nm0211667, n..."
3,tt0002885,From Dusk to Dawn,1913,[Drama],90,5.9,119,"[nm0191890, nm0201679, nm0204584, nm0254556, n..."
8,tt0006419,El beso de la muerte,1917,[Drama],72,4.1,24,"[nm0022279, nm0049367, nm0077222, nm0140054, n..."
17,tt0007983,Fear,1917,[Horror],72,6.1,283,"[nm0213638, nm0324553, nm0681620, nm0857350, n..."


##

##  Cheking data validation

In [9]:
# חישוב אחוז הערכים הקיימים (ולא החסרים)
completeness_percentage = (imdb_df.notnull().sum() / len(imdb_df)) * 100

print("Percentage of complete values:")
print("")
print(completeness_percentage.map('{:.2f}%'.format))


print("")


imdb_df.describe()

Percentage of complete values:

tconst             100.00%
primaryTitle       100.00%
startYear          100.00%
genres             100.00%
runtimeMinutes     100.00%
averageRating      100.00%
numVotes           100.00%
lead_actors_ids    100.00%
dtype: object



,startYear,runtimeMinutes,averageRating,numVotes
count,5000.0,5000.0,5000.000000,5000.0
mean,1973.1254,93.5114,5.926020,5557.0188
std,21.275512,18.358771,1.106773,62924.294182
min,1912.0,60.0,1.800000,5.0
25%,1958.0,84.0,5.300000,30.0
50%,1976.0,91.0,6.000000,97.0
75%,1991.0,100.0,6.700000,440.0
max,2009.0,298.0,9.500000,2599988.0


## The data present is valid, full and fits the required instructions.

##

##

# Collecting data columns from Wikipedia by Web Scraping #

## Plot Column:

In [11]:
def get_movie_plots(movie_data_list):
    # הזדהות כדפדפן
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    movie_list = []

    for tconst, movie_name, movie_year in movie_data_list:
        plot_short = "No plot found" 
        response = None 
        final_url = "" # משמש ללוגיקה הפנימית בלבד

        # מניפולציות על שם הסרט
        formatted_name = str(movie_name).replace(" ", "_")
        name_no_comma = str(movie_name).replace(",", "")
        formatted_no_comma = name_no_comma.replace(" ", "_")      
        alt_name = name_no_comma
        if alt_name.lower().startswith("the "):
            alt_name = re.sub(r'^The ', 'A ', alt_name, flags=re.IGNORECASE)
        elif alt_name.lower().startswith("a "):
            alt_name = re.sub(r'^A ', 'The ', alt_name, flags=re.IGNORECASE)
        
        # כתובות לניסיון
        urls_to_try = [
            f"https://en.wikipedia.org/wiki/{formatted_name}_({movie_year}_film)",
            f"https://en.wikipedia.org/wiki/{formatted_name}_(film)",
            f"https://en.wikipedia.org/wiki/{formatted_name}",
            f"https://en.wikipedia.org/wiki/{formatted_no_comma}",
            f"https://en.wikipedia.org/wiki/{alt_name.replace(' ', '_')}",
            f"https://sv.wikipedia.org/wiki/{formatted_name}"
        ]

        # חיפוש דף פעיל
        for target_url in urls_to_try:
            try:
                res = requests.get(target_url, headers=headers, timeout=5)
                if res.status_code == 200:
                    response = res
                    final_url = target_url
                    break
            except Exception:
                continue

        if not response:
            movie_list.append((tconst, "No plot found"))
            continue

        soup = BeautifulSoup(response.content, 'html.parser')
        content_div = soup.find('div', {'class': 'mw-parser-output'})
        
        def extract_plot(container):
            if not container: return None
            all_ps = container.find_all('p', recursive=False)
            for p in all_ps:
                t = re.sub(r'\[.*?\]', '', p.get_text().strip())
                if len(t) > 40:
                    sentences = t.split('. ')
                    res = ". ".join(sentences[:2]).strip()
                    return res if res.endswith('.') else res + "."
            return None

        # טיפול בפירושונים
        first_p = content_div.find('p') if content_div else None
        first_p_text = first_p.get_text().lower() if first_p else ""
        if ("refer to" in first_p_text) or ("kan syfta på" in first_p_text) or soup.find('.disambig'):
            links = content_div.find_all('a', href=True)
            for link in links:
                link_text = link.get_text().lower()
                if movie_name.lower() in link_text or "film" in link_text:
                    base_domain = "https://sv.wikipedia.org" if "sv.wikipedia.org" in final_url else "https://en.wikipedia.org"
                    new_url = base_domain + link['href']
                    try:
                        res = requests.get(new_url, headers=headers, timeout=5)
                        if res.status_code == 200:
                            soup = BeautifulSoup(res.content, 'html.parser')
                            content_div = soup.find('div', {'class': 'mw-parser-output'})
                            break
                    except: continue

        plot_short = extract_plot(content_div)

        # גיבוי אם נכשל
        if not plot_short or plot_short == "No plot found":
            all_ps = content_div.find_all('p', recursive=False) if content_div else []
            if len(all_ps) >= 2:
                backup_p = all_ps[1].get_text().strip()
                plot_short = backup_p[:250] if len(backup_p) > 10 else "No plot found"
            else:
                plot_short = "No plot found"

        # הוספה לרשימה - רק tconst ו-plot
        movie_list.append((tconst, plot_short))
        time.sleep(0.3)
        
    return movie_list

# הכנת הקלט מתוך ה-imdb_df הקיים שלך
movie_input = list(zip(imdb_df['tconst'], imdb_df['primaryTitle'], imdb_df['startYear']))[0:5000]

# הרצת הפונקציה
results = get_movie_plots(movie_input)

# יצירת ה-DataFrame עם 2 עמודות בלבד
df_results = pd.DataFrame(results, columns=['tconst', 'plot'])

# שמירה ל-CSV
df_results.to_csv('wiki_plots.csv', index=False, encoding='utf-8-sig')

KeyboardInterrupt: 

## Country and Language Columns:

In [ ]:
#עובד שפה ארץ

def get_movie_data(movie_data_list):
    movie_list = []
    headers = {
        'User-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    for tconst, movie_name, movie_year in movie_data_list:
        print(f"Searching: {movie_name}...")
        response = None

        formatted_name = str(movie_name).replace(" ", "_")
        urls_to_try = [
            f"https://en.wikipedia.org/wiki/{formatted_name}_({movie_year}_film)",
            f"https://en.wikipedia.org/wiki/{formatted_name}_(film)",
            f"https://en.wikipedia.org/wiki/{formatted_name}"
        ]

        for target_url in urls_to_try:
            try:
                res = requests.get(target_url, headers=headers, timeout=5)
                if res.status_code == 200 and "refer to:" not in res.text.lower():
                    response = res
                    break
            except:
                continue

        if not response:
            movie_list.append({
                'tconst': tconst, 
                'Language': 'Not Found', 
                'Country': 'Not Found',
                'Title': movie_name  # Title בסוף
            })
            continue

        soup = BeautifulSoup(response.content, 'html.parser')
        
        def get_infobox_val(label_text):
            tag = soup.find('th', string=re.compile(label_text, re.IGNORECASE))
            if tag:
                td = tag.find_next('td')
                return re.sub(r'\[.*?\]', '', td.get_text(" ", strip=True))
            return "N/A"

        movie_list.append({
            'tconst': tconst,
            'Language': get_infobox_val('Language'),
            'Country': get_infobox_val('Country'),
            'Title': movie_name  # Title בסוף
        })
        
        time.sleep(0.3) 

    return pd.DataFrame(movie_list)

# --- הרצה ---
movie_input = list(zip(imdb_df['tconst'], imdb_df['primaryTitle'], imdb_df['startYear']))[0:5000]
df_final = get_movie_data(movie_input)
df_final.to_csv('wiki_countryandlang.csv', index=False, encoding='utf-8-sig')

## Budget and Box Office Columns:

In [ ]:
##פעולה שתמיר ערכי טקטס עם סימן דולר\יורו או מיליון- למספר עשרוני
def clean_currency_to_numeric(df, columns_to_fix):
    def convert_text_to_number(text):
        if pd.isna(text) or isinstance(text, (int, float)):
            return text
        text = str(text).lower()
        # ניקוי סימנים: $, פסיקים, והערות שוליים
        text = re.sub(r'[$,\[.*?\]]', '', text)
        # קביעת המכפיל
        multiplier = 1
        if 'million' in text:
            multiplier = 1_000_000
        elif 'billion' in text:
            multiplier = 1_000_000_000
        try:
            # מציאת כל המספרים (כדי לזהות טווח כמו 10-15)
            numbers = re.findall(r"\d*\.\d+|\d+", text)
            if not numbers:
                return None
            
         # המרה ל-float
            nums_as_floats = [float(n) for n in numbers]
            # חישוב ממוצע (אם יש מספר אחד הממוצע הוא המספר עצמו)
            avg_number = sum(nums_as_floats) / len(nums_as_floats)
            return avg_number * multiplier
        except:
            return None

    new_df = df.copy()
    for col in columns_to_fix:
        if col in new_df.columns:
            new_df[col] = new_df[col].apply(convert_text_to_number)
    return new_df

In [ ]:
#המשיכה עצמה
def get_movie_data(movie_data_list):
    movie_list = []
    headers = {
        'User-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    # מוסיפים enumerate מסביב לרשימה כדי לקבל את המשתנה index
    for index, (tconst, movie_name, movie_year) in enumerate(movie_data_list, start=1):
        #print(f"שורה : {index} / 2500")
        response = None
        final_url = None
        
        # לוגיקת הכנת שמות מהקוד השני
        formatted_name = str(movie_name).replace(" ", "_")
        name_no_comma = str(movie_name).replace(",", "")
        formatted_no_comma = name_no_comma.replace(" ", "_")      
        alt_name = name_no_comma
        if alt_name.lower().startswith("the "):
            alt_name = re.sub(r'^The ', 'A ', alt_name, flags=re.IGNORECASE)
        elif alt_name.lower().startswith("a "):
            alt_name = re.sub(r'^A ', 'The ', alt_name, flags=re.IGNORECASE)

        # עץ הכתובות מהקוד השני
        urls_to_try = [
            ("שם + שנה", f"https://en.wikipedia.org/wiki/{formatted_name}_({movie_year}_film)"),
            ("שם + (film)", f"https://en.wikipedia.org/wiki/{formatted_name}_(film)"),
            ("שם רגיל", f"https://en.wikipedia.org/wiki/{formatted_name}"),
            ("ללא פסיק", f"https://en.wikipedia.org/wiki/{formatted_no_comma}"),
            ("The/A החלפה", f"https://en.wikipedia.org/wiki/{alt_name.replace(' ', '_')}"),
            ("ויקיפדיה שוודית", f"https://sv.wikipedia.org/wiki/{formatted_name}")
        ]

        for desc, target_url in urls_to_try:
            try:
                #print(f"בודק אפשרות {desc}: {target_url}")
                res = requests.get(target_url, headers=headers, timeout=5)
                
                if res.status_code == 200:
                    soup = BeautifulSoup(res.content, 'html.parser')
                    content_div = soup.find('div', {'class': 'mw-parser-output'})
                    
                    # לוגיקת בדיקת פירושונים מהקוד השני
                    first_p = content_div.find('p') if content_div else None
                    first_p_text = first_p.get_text().lower() if first_p else ""
                    
                    if ("refer to" in first_p_text) or ("kan syfta på" in first_p_text) or soup.find('.disambig'):
                        #print(f"(!) זוהה דף פירושונים בקישור: {target_url}")
                        links = content_div.find_all('a', href=True) if content_div else []
                        found_inner = False
                        for link in links:
                            link_text = link.get_text().lower()
                            # חיפוש קישור עם שם הסרט או המילה film כפי שמופיע בקוד השני
                            if movie_name.lower() in link_text or "film" in link_text:
                                base_domain = "https://sv.wikipedia.org" if "sv.wikipedia.org" in target_url else "https://en.wikipedia.org"
                                new_url = base_domain + link['href']
                                #print(f"--> החלטה: כניסה לקישור הראשון ב-mw-parser-output: {new_url}")
                                res_link = requests.get(new_url, headers=headers, timeout=5)
                                if res_link.status_code == 200:
                                    response = res_link
                                    final_url = new_url
                                    found_inner = True
                                    break
                        if found_inner: break
                        else: 
                            #print("x לא נמצא קישור מתאים בתוך הפירושונים, ממשיך.")
                            continue
                    
                    # אם נמצא דף תקין שאינו פירושונים
                    #print(f"V נמצא דף ישיר: {target_url}")
                    response = res
                    final_url = target_url
                    break
                else:
                    print("")
            except Exception as e:
                #print(f"error שגיאה: {e}")
                continue

        # שליפת נתונים (מתוך הקוד הראשון)
        if not response:
            #print(f"FAILED: לא נמצא דף עבור {movie_name}")
            movie_list.append({
                'tconst': tconst, 'budget': 'Not Found', 'boxoffice': 'Not Found', 'Title': movie_name, 'URL': 'N/A'
            })
            continue

        soup = BeautifulSoup(response.content, 'html.parser')
        
        def get_infobox_val(label_text):
            tag = soup.find('th', string=re.compile(label_text, re.IGNORECASE))
            if tag:
                td = tag.find_next('td')
                val = td.get_text(" ", strip=True)
                return re.sub(r'\[.*?\]', '', val).strip() 
            return "N/A"

        budget_val = get_infobox_val('Budget')
        box_val = get_infobox_val('Box office')
        
        #print(f"נשלף בהצלחה -> Budget: {budget_val} | Box Office: {box_val}")

        movie_list.append({
            'tconst': tconst,
            'budget': budget_val,
            'boxoffice': box_val,
            'Title': movie_name,
            'URL': final_url
        })
        
        time.sleep(0.3) 

    return pd.DataFrame(movie_list)

# הרצה על הטווח שציינת
movie_input = list(zip(imdb_df['tconst'], imdb_df['primaryTitle'], imdb_df['startYear']))[2500:5000]#שם קובץ
results = get_movie_data(movie_input)
results

if isinstance(results, list):
    results = pd.DataFrame(results)
df_final2 = clean_currency_to_numeric(results, ['budget', 'boxoffice'])
df_final2.to_csv('wiki_budget_box3.csv', index=False, encoding='utf-8-sig')
df_final2.head()

### Finally we got all the data and save it in csv files!

## merging all fiels & saving one complete CSV file

In [12]:
df1 = pd.read_csv('wiki_budget_box1.csv')
df2 = pd.read_csv('wiki_budget_box3.csv')

# איחוד הטבלאות אחת מתחת לשניה
wiki_budget_box = pd.concat([df1, df2], ignore_index=True)

# שמירה לקובץ חדש
wiki_budget_box.to_csv('wiki_budget_box.csv', index=False)

In [15]:
df1 = pd.read_csv('imdb_pd.csv')
df2 = pd.read_csv('wiki_plots.csv')
df3 = pd.read_csv('wiki_countryandlang.csv')
df4 = pd.read_csv('wiki_budget_box.csv')


all_data_df = df1.merge(df2, on='tconst', how='left') \
                   .merge(df3, on='tconst', how='left') \
                   .merge(df4, on='tconst', how='left')
all_data_df.head()

,tconst,primaryTitle,startYear,genres,runtimeMinutes,averageRating,numVotes,lead_actors_ids,plot,Language,Country,Title_x,budget,boxoffice,Title_y,URL
0,tt0002199,From the Manger to the Cross,1912,"['Biography', 'Drama']",71,5.9,712,"['nm0087381', 'nm0245769', 'nm0310155', 'nm039...",From the Manger to the Cross or Jesus of Nazar...,Silent (English intertitles ),United States,From the Manger to the Cross,35000.0,1000000.0,From the Manger to the Cross,https://en.wikipedia.org/wiki/From_the_Manger_...
1,tt0002820,East Lynne,1913,['Drama'],69,6.2,22,"['nm0287085', 'nm0666835', 'nm0604658', 'nm021...",East Lynne is a 1913 British silent drama film...,English,United Kingdom,East Lynne,NaN,NaN,East Lynne,https://en.wikipedia.org/wiki/East_Lynne_(1913...
2,tt0002885,From Dusk to Dawn,1913,['Drama'],90,5.9,119,"['nm0191890', 'nm0201679', 'nm0204584', 'nm025...",From Dusk till Dawn is a 1996 American horror...,Not Found,Not Found,From Dusk to Dawn,NaN,NaN,From Dusk to Dawn,https://en.wikipedia.org/wiki/From_Dusk_to_Dawn
3,tt0006419,El beso de la muerte,1917,['Drama'],72,4.1,24,"['nm0022279', 'nm0049367', 'nm0077222', 'nm014...",No plot found,Not Found,Not Found,El beso de la muerte,NaN,NaN,El beso de la muerte,NaN
4,tt0007983,Fear,1917,['Horror'],72,6.1,283,"['nm0213638', 'nm0324553', 'nm0681620', 'nm085...",Fear (German: Furcht) is a 1917 German silent ...,Silent German intertitles,Germany,Fear,NaN,NaN,Fear,https://en.wikipedia.org/wiki/Fear_(1917_film)


In [16]:
all_data_df = all_data_df.drop(['Title_x', 'Title_y', 'URL'], axis=1)
all_data_df.to_csv('all_data.csv', index=False)

## Data checking:

In [17]:


# חישוב אחוז הערכים הקיימים (ולא החסרים)
completeness_percentage = (all_data_df.notnull().sum() / len(all_data_df)) * 100

print("Percentage of complete values:")
print("")
print(completeness_percentage.map('{:.2f}%'.format))


print("")
print("")
print("")


cols_to_check = [
    'tconst', 'primaryTitle', 'startYear', 'genres', 'runtimeMinutes', 
    'averageRating', 'numVotes', 'lead_actors_ids', 'Language', 
    'Country', 'plot', 'budget', 'boxoffice'
]

print("Column Name | Data Type (First Row)")
print("-" * 40)

for col in cols_to_check:
    if col in all_data_df.columns:
        first_value_type = type(all_data_df[col].iloc[0])
        print(f"{col:<20} | {first_value_type}")
    else:
        print(f"{col:<20} | Column not found!")
        
        
        
        

print("")

all_data_df.describe()




Percentage of complete values:

tconst             100.00%
primaryTitle       100.00%
startYear          100.00%
genres             100.00%
runtimeMinutes     100.00%
averageRating      100.00%
numVotes           100.00%
lead_actors_ids    100.00%
plot               100.00%
Language            92.94%
Country             87.28%
budget              11.94%
boxoffice           13.42%
dtype: object



Column Name | Data Type (First Row)
----------------------------------------
tconst               | <class 'str'>
primaryTitle         | <class 'str'>
startYear            | <class 'numpy.int64'>
genres               | <class 'str'>
runtimeMinutes       | <class 'numpy.int64'>
averageRating        | <class 'numpy.float64'>
numVotes             | <class 'numpy.int64'>
lead_actors_ids      | <class 'str'>
Language             | <class 'str'>
Country              | <class 'str'>
plot                 | <class 'str'>
budget               | <class 'numpy.float64'>
boxoffice            | <class 'nump

,startYear,runtimeMinutes,averageRating,numVotes,budget,boxoffice
count,5000.000000,5000.000000,5000.000000,5.000000e+03,5.970000e+02,6.710000e+02
mean,1973.125400,93.511400,5.926020,5.557019e+03,7.714735e+09,4.777362e+10
std,21.275512,18.358771,1.106773,6.292429e+04,1.221957e+11,5.344905e+11
min,1912.000000,60.000000,1.800000,5.000000e+00,2.000000e+00,0.000000e+00
25%,1958.000000,84.000000,5.300000,3.000000e+01,5.000000e+05,1.050000e+06
50%,1976.000000,91.000000,6.000000,9.700000e+01,5.000000e+06,1.500000e+07
75%,1991.000000,100.000000,6.700000,4.400000e+02,2.500000e+07,1.250000e+08
max,2009.000000,298.000000,9.500000,2.599988e+06,2.805180e+12,1.112881e+13


#

#                                   DONE!